# School Census (INEP): how the raw layer is obtained

**Publisher:** Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira (INEP)  
**Official source:** <https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar>

Yearly School Census microdata at school level: schools, classes, teachers, enrolments and infrastructure, with the historical series recovered from 1995. Student-level and teacher-level microdata are not released.

**Grain and keys:** School by year for the school tables; the semantic tables aggregate to school-year and municipality-year. The 2025 edition changed the enrolment file from student level to school-level aggregates.

**Released as:**

- `censo-escolar-analytics`: semantic (10 tables)
- `censo-escolar-raw-trusted-part-1`: raw (1 tables)
- `censo-escolar-raw-trusted-part-2`: trusted (1 tables)
- `censo-escolar-raw-trusted-part-3`: raw, trusted (30 tables)

Every table and column is documented in the [interactive data map](https://rangeltech.net/datamap/). Source code and the other notebooks: [https://github.com/LucasRangelSSouza/brazil-public-data-map](https://github.com/LucasRangelSSouza/brazil-public-data-map).

## Status of this notebook

**Not executed end to end.** The download cells below use URLs recorded in the released tables, but the publisher's download host did not respond when this notebook was written, so the outputs are empty. Run it from your own network.

In [ ]:
import json, time
import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; public-data-map-notebook; +https://github.com/LucasRangelSSouza/brazil-public-data-map)"}

def get(url, retries=4, timeout=90, **kw):
    """GET with exponential backoff. Public portals answer 429 or reset connections under load."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout, **kw)
            if r.status_code in (429, 500, 502, 503, 504):
                raise requests.HTTPError(f"HTTP {r.status_code}")
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

## 1. The yearly microdata

INEP publishes one ZIP per year on the School Census page. The file names follow `microdados_censo_escolar_<year>.zip`; read the page to confirm the current link for a year. Inside, the CSV has one row per school (the released tables do not include student-level or teacher-level microdata) and the ZIP carries the official data dictionary.

In [ ]:
YEAR = 2023
PAGE = "https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar"
URL = f"https://download.inep.gov.br/dados_abertos/microdados_censo_escolar_{YEAR}.zip"   # pattern published on the page above
print(PAGE); print(URL)

In [ ]:
from pathlib import Path
import zipfile
target = Path("downloads") / f"censo_{YEAR}.zip"; target.parent.mkdir(exist_ok=True)
try:
    with get(URL, stream=True) as r, open(target, "wb") as fh:
        for chunk in r.iter_content(1 << 20):
            fh.write(chunk)
    print([n for n in zipfile.ZipFile(target).namelist() if n.lower().endswith((".csv", ".pdf", ".xlsx"))][:12])
except Exception as exc:
    print("download failed (check the page for the current link):", exc)

## 2. Formats change between years

The census files changed layout over the years: older editions use one file per module, recent ones a single microdata CSV, and the 2025 enrolment file is school-level aggregates. The released `inep_censo_escolar_colunas_fonte` and `inep_censo_escolar_dicionario_campos` tables record the column mapping between editions.

## Compare with the released tables

The released files keep the field names of the source in `raw` and apply consistent typing and snake_case names in `trusted`. Details per column are in the [data map](https://rangeltech.net/datamap/#/subject/censo-escolar).

In [ ]:
# Read one released table without downloading the whole dataset. Public datasets need no login.
import kagglehub, pandas as pd
path = kagglehub.dataset_download("lucasrangelss/censo-escolar-raw-trusted-part-3", path="trusted__inep_censo_escolar_colunas_fonte.parquet")
pd.read_parquet(path).head()